In [ ]:
import logging
from IPython.display import Markdown
from library.circuitry import Circuitry
from library.magic_state_cultivation import MagicStateCultivation
from library.common import Pauli
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.stim import simulate, sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 7

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
scenarios: dict[str, Circuitry] = dict()
point: int = 0

In [ ]:
if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
msc = MagicStateCultivation(
    injection=SteaneCodePatch.Injection.S, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
)

msc.append_preparation()

msc.circuitry.append_observable(0, "Y_OBSERVABLE_PREPARED", msc.steane.logical(Pauli.Y))

msc.circuitry.detectors_report()

scenario = "Prepared"
scenarios[scenario] = msc.circuitry
msc.circuitry.to_file(FILEROOT + f".point{point}.prepared")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

In [ ]:
msc = MagicStateCultivation(
    injection=SteaneCodePatch.Injection.S, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
)

msc.append_preparation()
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(rnd)

msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)

msc.circuitry.append_observable(0, "Y_OBSERVABLE_SUPERDENSE", msc.steane.logical(Pauli.Y))

msc.circuitry.detectors_report()

scenario = f"SDCx{SUPERDENSE_ROUNDS}"
scenarios[scenario] = msc.circuitry
msc.circuitry.to_file(FILEROOT + f".point{point}.superdense")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

In [ ]:
msc = MagicStateCultivation(
    injection=SteaneCodePatch.Injection.S, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
)

msc.append_preparation()
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(rnd)
msc.append_cultivation()

msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)

msc.circuitry.append_observable(0, "Y_OBSERVABLE_CULTIVATION", msc.steane.logical(Pauli.Y))

msc.circuitry.detectors_report()

scenario = f"Double-checked"
scenarios[scenario] = msc.circuitry
msc.circuitry.to_file(FILEROOT + f".point{point}.double-check-s")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

In [ ]:
msc = MagicStateCultivation(
    injection=SteaneCodePatch.Injection.S, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
)

msc.append_preparation()
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(rnd)
msc.append_cultivation()
msc.append_teleportation(TELEPORT_ROUNDS)

msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)

msc.circuitry.append_observable(
    0, "Y_OBSERVABLE_TELEPORTED", msc.source.logical(Pauli.Y),
    *["JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"]
)

msc.circuitry.detectors_report()

scenario = f"Teleported"
scenarios[scenario] = msc.circuitry
msc.circuitry.to_file(FILEROOT + f".point{point}.teleported")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

In [ ]:
msc = MagicStateCultivation(
    injection=SteaneCodePatch.Injection.S, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
)

msc.append_preparation()
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(rnd)
msc.append_cultivation()
msc.append_teleportation(TELEPORT_ROUNDS)
msc.append_expansion()

msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)

msc.circuitry.append_observable(
    0, "Y_OBSERVABLE_EXPANDED", msc.target.logical(Pauli.Y),
    *["JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"]
)

msc.circuitry.detectors_report()

scenario = f"Expanded"
scenarios[scenario] = msc.circuitry
msc.circuitry.to_file(FILEROOT + f".point{point}.expanded")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [Corrected $\overline{\mathbf{Y}}$]"
sample(scenarios, title=title, label="Point", shots=1e6, correction=True, fontsize=10)

In [ ]:
simulate(
    scenarios, title, label="Point",
    postselection=True, shots=1e6, minimal_noise=-6,
    figsize=(11, 4.5), num_workers=7,
)